In [1]:
import sys
!{sys.executable} -m pip install azure-ai-projects azure-ai-agents azure-identity --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = "your_project_endpoint"

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)
print("Csatlakozva az AI Foundry projekthez")

Csatlakozva az AI Foundry projekthez


In [ ]:
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from msrest.authentication import ApiKeyCredentials
import json

PREDICTION_ENDPOINT = "your_pred_endpoint"
PREDICTION_KEY = "your_pred_key"
CV_PROJECT_ID = "your_project_id"
PUBLISHED_NAME = "your_published_name"

DX_FULL_NAMES = {
    "nv": "melanocytic nevus (common mole, generally benign)",
    "mel": "melanoma (a serious form of skin cancer)",
    "bkl": "benign keratosis-like lesion",
    "bcc": "basal cell carcinoma (a common, usually slow-growing skin cancer)",
    "akiec": "actinic keratosis / intraepithelial carcinoma (a precancerous lesion)",
    "vasc": "vascular lesion (e.g. angioma)",
    "df": "dermatofibroma (a benign skin growth)"
}

prediction_credentials = ApiKeyCredentials(in_headers={"Prediction-key": PREDICTION_KEY})
predictor = CustomVisionPredictionClient(PREDICTION_ENDPOINT, prediction_credentials)

def analyze_skin_lesion_image(image_path: str) -> str:
    """
    Analyzes a skin lesion image using the trained Custom Vision model.
    Returns the top predicted categories with their confidence scores.
    """
    with open(image_path, "rb") as image_data:
        results = predictor.classify_image(CV_PROJECT_ID, PUBLISHED_NAME, image_data.read())

    predictions = sorted(results.predictions, key=lambda p: p.probability, reverse=True)[:3]

    output = []
    for p in predictions:
        output.append({
            "category_code": p.tag_name,
            "category_full_name": DX_FULL_NAMES.get(p.tag_name, p.tag_name),
            "confidence": round(p.probability * 100, 1)
        })

    return json.dumps(output)

In [ ]:
def explain_abcde_rule() -> str:
    """
    Returns an explanation of the ABCDE rule, a common educational
    self-examination guideline for skin lesions, used by dermatologists
    to describe warning signs worth having checked.
    """
    info = {
        "A - Asymmetry": "One half of the lesion doesn't match the other half.",
        "B - Border": "Edges are irregular, ragged, notched, or blurred.",
        "C - Color": "Color is not uniform; may include shades of brown, black, or patches of pink, red, white, or blue.",
        "D - Diameter": "Larger than about 6mm (roughly the size of a pencil eraser), though some can be smaller.",
        "E - Evolving": "The lesion has changed in size, shape, color, or texture over time, or new symptoms like itching or bleeding appear."
    }
    return json.dumps(info)

In [4]:
SYSTEM_INSTRUCTIONS = """
You are an educational assistant that helps users understand general characteristics of skin lesions, based on an image analysis tool. You are NOT a medical device and you NEVER provide a diagnosis.

Rules you must always follow:
1. When a user shares a skin lesion image, use the analyze_skin_lesion_image tool to get the model's predictions.
2. Explain the results in plain, calm, reassuring language. Mention that this is an automated pattern-recognition tool trained on a limited dataset, not a diagnostic tool.
3. NEVER state or imply a diagnosis. Use phrasing like "the tool's top prediction is X, but this is not a medical diagnosis."
4. ALWAYS end your response by recommending the user consult a dermatologist or doctor for an accurate assessment, especially if the top prediction is melanoma, basal cell carcinoma, or actinic keratosis (potentially serious categories).
5. If the user provides additional context (age, how long the lesion has been present, whether it has changed), you may mention that these factors are also important for a doctor to evaluate, but do not attempt to interpret them yourself.
6. Keep your response concise: 3-5 sentences.
7. If relevant, you may use the explain_abcde_rule tool to give the user general educational context about what dermatologists look for, but always frame it as general education, not an assessment of their specific case.
"""

In [ ]:
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import FunctionTool, ToolSet
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = "your_project_endpoint"

agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

functions = FunctionTool(functions={analyze_skin_lesion_image, explain_abcde_rule})
toolset = ToolSet()
toolset.add(functions)
agents_client.enable_auto_function_calls(toolset)

agent = agents_client.create_agent(
    model="gpt-5-mini",
    name="skin-lesion-educational-agent",
    instructions=SYSTEM_INSTRUCTIONS,
    toolset=toolset
)

print(f"Agent létrehozva: {agent.id}")

Agent létrehozva: asst_pG2nUMpo0ttRcPKmac8aQ1kH


In [9]:
import os

part1_path = "./ham10000_data/HAM10000_images_part_1"
example_images = os.listdir(part1_path)
test_image_path = os.path.join(part1_path, example_images[0])

print(f"Teszt kép: {test_image_path}")

Teszt kép: ./ham10000_data/HAM10000_images_part_1/ISIC_0024306.jpg


In [ ]:
import pandas as pd
metadata = pd.read_csv("./ham10000_data/HAM10000_metadata.csv")
mel_examples = metadata[metadata["dx"] == "mel"]["image_id"].tolist()

test_image_path_2 = f"./ham10000_data/HAM10000_images_part_1/{mel_examples[0]}.jpg"
if not os.path.exists(test_image_path_2):
    test_image_path_2 = f"./ham10000_data/HAM10000_images_part_2/{mel_examples[0]}.jpg"

print(f"Teszt kép (mel): {test_image_path_2}")

Teszt kép (mel): ./ham10000_data/HAM10000_images_part_1/ISIC_0025964.jpg


In [14]:
thread = agents_client.threads.create()

user_message = f"""
I have a skin lesion I'd like to understand better. Here's some context:
- Age: 45
- Location on body: forearm
- Present for approximately 6 months
- Has not noticeably changed in size or color

Please analyze the image at this path: {test_image_path_2}
"""

agents_client.messages.create(thread_id=thread.id, role="user", content=user_message)

run = agents_client.runs.create_and_process(
    thread_id=thread.id,
    agent_id=agent.id
)

print(f"Run státusz: {run.status}")

Run státusz: completed


In [15]:
messages = agents_client.messages.list(thread_id=thread.id)
for msg in messages:
    if msg.content:
        print(f"[{msg.role}]: {msg.content[0].text.value}")
        print("---")

[assistant]: The tool's top prediction is melanoma (43.0% confidence), followed by melanocytic nevus (34.7%) and benign keratosis-like lesion (13.4%), but this is not a medical diagnosis. This is an automated pattern-recognition model trained on a limited dataset and cannot replace an in-person clinical evaluation. Your age, forearm location, how long it’s been present, and whether it has changed are all important details for a clinician to consider. Because the top prediction is melanoma, please consult a dermatologist or doctor for an accurate assessment.
---
[user]: 
I have a skin lesion I'd like to understand better. Here's some context:
- Age: 45
- Location on body: forearm
- Present for approximately 6 months
- Has not noticeably changed in size or color

Please analyze the image at this path: ./ham10000_data/HAM10000_images_part_1/ISIC_0025964.jpg

---
